In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('/rfm_data.csv')
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

print(df['order_purchase_timestamp'].min(), df['order_purchase_timestamp'].max())

2016-10-03 09:44:50 2018-08-29 15:00:37


In [3]:
cutoff  = pd.Timestamp('2018-03-01')
horizon = cutoff + pd.Timedelta(days=90)

past   = df[df['order_purchase_timestamp'] < cutoff]
future = df[(df['order_purchase_timestamp'] >= cutoff) &
            (df['order_purchase_timestamp'] <  horizon)]

print("Past rows  :", len(past))
print("Future rows:", len(future))

Past rows  : 60131
Future rows: 21084


In [4]:
feat = past.groupby('customer_unique_id').agg(
    recency=('order_purchase_timestamp', lambda x: (cutoff - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('payment_value', 'sum')
).reset_index()

feat['returned'] = feat['customer_unique_id'].isin(future['customer_unique_id']).astype(int)

print(feat['returned'].value_counts(normalize=True))

returned
0    0.993048
1    0.006952
Name: proportion, dtype: float64


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

X = feat[['recency', 'frequency', 'monetary']].copy()
X['monetary'] = np.log1p(X['monetary'])
y = feat['returned']

print("Total customers:", len(y), "| Returned:", y.sum())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    'Logistic Regression': make_pipeline(StandardScaler(),
                            LogisticRegression(class_weight='balanced', max_iter=1000)),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=6,
                            class_weight='balanced', random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    print(f"\n=== {name} ===")
    print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
    print("PR-AUC :", round(average_precision_score(y_test, proba), 3))
    print(classification_report(y_test, model.predict(X_test), zero_division=0))

Total customers: 55524 | Returned: 386

=== Logistic Regression ===
ROC-AUC: 0.609
PR-AUC : 0.052
              precision    recall  f1-score   support

           0       0.99      0.59      0.74     11028
           1       0.01      0.56      0.02        77

    accuracy                           0.59     11105
   macro avg       0.50      0.58      0.38     11105
weighted avg       0.99      0.59      0.74     11105


=== Random Forest ===
ROC-AUC: 0.616
PR-AUC : 0.011
              precision    recall  f1-score   support

           0       0.99      0.88      0.93     11028
           1       0.01      0.23      0.02        77

    accuracy                           0.87     11105
   macro avg       0.50      0.55      0.48     11105
weighted avg       0.99      0.87      0.92     11105



## Model comparison

Logistic Regression and Random Forest reach similar ROC-AUC (0.61 vs 0.62), but
Logistic Regression ranks likely repurchasers better (PR-AUC 0.052 vs 0.011; random
baseline is about 0.007). With only 77 positive cases in the test set, differences
are small and uncertain.

**Chosen model:** Logistic Regression (better PR-AUC, simpler, easier to explain).

**Limitation:** RFM features alone carry weak signal. Next step: add delivery delay,
review score and freight value from other Olist tables.